In [ ]:
from pyspark.sql import functions as F

S = "Gold"
BASE = "abfss://718e8176-5d40-4a9c-88ff-50ac97ac49ba@onelake.dfs.fabric.microsoft.com/981fbe98-2f01-41d8-bf2f-a85e5cd9e2a2"
META_SILVER = f"{BASE}/Files/Silver/meta_ads"

def ensure_thumbnail(df, platform_hint=None):
    cols = set(df.columns)
    if "final_urls" in cols and "thumbnail_url" not in cols:
        df = df.withColumnRenamed("final_urls", "thumbnail_url")
    elif "final_urls" in cols and "thumbnail_url" in cols:
        df = df.drop("final_urls")
    if "thumbnail_url" not in df.columns:
        df = df.withColumn("thumbnail_url", F.lit(None).cast("string"))
    return df

# Load Meta thumbs
ads = (
    spark.read.format("delta").load(f"{META_SILVER}/silver_meta_ads")
    .select(F.col("ad_id").alias("_ad_id"), F.col("thumbnail_url").alias("_thumb"))
    .dropDuplicates(["_ad_id"])
)
print("ads thumbs", ads.where(F.col("_thumb").isNotNull()).count())

# META
meta = ensure_thumbnail(spark.table(f"{S}.rpt_meta_ad_performance_daily"))
meta = (
    meta.drop("thumbnail_url")
    .join(ads, meta.ad_id == ads._ad_id, "left")
    .withColumn("thumbnail_url", F.col("_thumb"))
    .drop("_ad_id", "_thumb")
)
print("meta", meta.count(), "thumb", meta.where(F.col("thumbnail_url").isNotNull()).count())
meta.write.format("delta").mode("overwrite").option("overwriteSchema","true").partitionBy("full_date").saveAsTable(f"{S}.rpt_meta_ad_performance_daily")
print("meta written")

# GOOGLE
g = ensure_thumbnail(spark.table(f"{S}.rpt_google_ad_performance_daily"))
g = g.withColumn("thumbnail_url", F.lit(None).cast("string"))
print("google", g.count(), "has final_urls", "final_urls" in g.columns)
g.write.format("delta").mode("overwrite").option("overwriteSchema","true").partitionBy("full_date").saveAsTable(f"{S}.rpt_google_ad_performance_daily")
print("google written")

# UNIFIED
u = ensure_thumbnail(spark.table(f"{S}.rpt_unified_ad_performance"))
u = (
    u.drop("thumbnail_url")
    .join(ads, u.ad_id == ads._ad_id, "left")
    .withColumn(
        "thumbnail_url",
        F.when(F.col("platform") == "meta", F.col("_thumb")).otherwise(F.lit(None).cast("string")),
    )
    .drop("_ad_id", "_thumb")
)
assert "final_urls" not in u.columns
assert "thumbnail_url" in u.columns
print("unified", u.count(), "thumb", u.where(F.col("thumbnail_url").isNotNull()).count())
u.write.format("delta").mode("overwrite").option("overwriteSchema","true").partitionBy("platform","full_date").saveAsTable(f"{S}.rpt_unified_ad_performance")
print("unified written")

spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_ad_performance AS SELECT * FROM {S}.rpt_unified_ad_performance")
spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_unified_ad_performance AS SELECT * FROM {S}.rpt_unified_ad_performance")
spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_meta_ad_performance AS SELECT * FROM {S}.rpt_meta_ad_performance_daily")
spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_google_ad_performance AS SELECT * FROM {S}.rpt_google_ad_performance_daily")

# refresh rollup views without referencing final_urls
spark.sql(f'''
CREATE OR REPLACE VIEW {S}.vw_adset_performance AS
SELECT platform, full_date, year, month, month_name, day_name,
  account_id, MAX(account_name) AS account_name,
  campaign_id, MAX(campaign_name) AS campaign_name,
  adset_id, MAX(adset_name) AS adset_name, MAX(adset_status) AS adset_status,
  MAX(optimization_goal) AS optimization_goal,
  MAX(age_range) AS age_range, MAX(geo_cities) AS geo_cities, MAX(geo_regions) AS geo_regions,
  SUM(impressions) AS impressions, SUM(reach) AS reach, SUM(clicks) AS clicks,
  SUM(spend) AS spend, SUM(leads) AS leads,
  CASE WHEN SUM(clicks) > 0 THEN SUM(spend)/SUM(clicks) ELSE NULL END AS cpc,
  CASE WHEN SUM(impressions) > 0 THEN (SUM(spend)/SUM(impressions))*1000 ELSE NULL END AS cpm,
  CASE WHEN SUM(leads) > 0 THEN SUM(spend)/SUM(leads) ELSE NULL END AS cost_per_lead,
  COUNT(DISTINCT ad_id) AS ad_count
FROM {S}.rpt_unified_ad_performance
GROUP BY platform, full_date, year, month, month_name, day_name, account_id, campaign_id, adset_id
''')
spark.sql(f'''
CREATE OR REPLACE VIEW {S}.vw_campaign_performance AS
SELECT platform, full_date, year, month, month_name, day_name,
  account_id, MAX(account_name) AS account_name,
  campaign_id, MAX(campaign_name) AS campaign_name,
  MAX(campaign_status) AS campaign_status,
  MAX(campaign_channel_or_objective) AS campaign_channel_or_objective,
  MAX(daily_budget_inr) AS daily_budget_inr,
  SUM(impressions) AS impressions, SUM(reach) AS reach, SUM(clicks) AS clicks,
  SUM(spend) AS spend, SUM(leads) AS leads,
  CASE WHEN SUM(clicks) > 0 THEN SUM(spend)/SUM(clicks) ELSE NULL END AS cpc,
  CASE WHEN SUM(impressions) > 0 THEN (SUM(spend)/SUM(impressions))*1000 ELSE NULL END AS cpm,
  CASE WHEN SUM(leads) > 0 THEN SUM(spend)/SUM(leads) ELSE NULL END AS cost_per_lead,
  COUNT(DISTINCT adset_id) AS adset_count, COUNT(DISTINCT ad_id) AS ad_count
FROM {S}.rpt_unified_ad_performance
GROUP BY platform, full_date, year, month, month_name, day_name, account_id, campaign_id
''')

cols = spark.table(f"{S}.rpt_unified_ad_performance").columns
print("final_urls?", "final_urls" in cols, "thumbnail_url?", "thumbnail_url" in cols)
spark.table(f"{S}.vw_ad_performance").select("platform","ad_name","thumbnail_url").where(F.col("thumbnail_url").isNotNull()).show(3, truncate=50)
print("RENAME_THUMBNAIL_URL_COMPLETE")

